In [1]:
import pyroll.wusatowski_spreading
import inspect

# Find where the package is installed
print(inspect.getfile(pyroll.wusatowski_spreading))

C:\Users\keert\anaconda3\Lib\site-packages\pyroll\wusatowski_spreading.py


In [2]:
# Read the actual wusatowski plugin source code
with open(r"C:\Users\keert\anaconda3\Lib\site-packages\pyroll\wusatowski_spreading.py", "r") as f:
    content = f.read()
print(content)

import logging

from pyroll.core import SymmetricRollPass, RollPass, ThreeRollPass, Hook

VERSION = "3.0.0"
PILLAR_MODEL_LOADED = False

SymmetricRollPass.wusatowski_temperature_coefficient = Hook[float]()
"""Temperature correction factor a for Wusatowski's spread equation."""

SymmetricRollPass.wusatowski_velocity_coefficient = Hook[float]()
"""Velocity correction factor c for Wusatowski's spread equation."""

SymmetricRollPass.wusatowski_material_coefficient = Hook[float]()
"""Material correction factor d for Wusatowski's spread equation."""

SymmetricRollPass.wusatowski_friction_coefficient = Hook[float]()
"""Friction correction factor f for Wusatowski's spread equation."""

SymmetricRollPass.wusatowski_exponent = Hook[float]()
"""Exponent w for Wusatowski's spread equation."""


@SymmetricRollPass.wusatowski_temperature_coefficient
def wusatowski_temperature_coefficient(self: SymmetricRollPass):
    return 1


@SymmetricRollPass.wusatowski_velocity_coefficient
def wusatowski_veloci

In [3]:
import os
print(os.getcwd())

C:\Users\keert\Untitled Folder


In [7]:
plugin_code = """
import numpy as np
import logging
from pyroll.core import SymmetricRollPass, Hook

VERSION = "0.1.0"

# New hooks — quantities our plugin computes
SymmetricRollPass.hitchcock_constant = Hook[float]()
SymmetricRollPass.flattened_roll_radius = Hook[float]()
SymmetricRollPass.flattening_ratio = Hook[float]()


@SymmetricRollPass.hitchcock_constant
def hitchcock_constant(self: SymmetricRollPass):
    E_roll  = 210e9
    nu_roll = 0.3
    return 16 * (1 - nu_roll**2) / (np.pi * E_roll)


@SymmetricRollPass.flattened_roll_radius
def flattened_roll_radius(self: SymmetricRollPass):
    R  = self.roll.nominal_radius
    F  = self.roll_force
    b  = self.in_profile.width
    dh = self.in_profile.height - self.gap

    if dh <= 0:
        logging.getLogger(__name__).warning(
            f"Non-positive draft in {self.label} — returning nominal radius."
        )
        return R

    return R * (1 + self.hitchcock_constant * F / (b * dh))


@SymmetricRollPass.flattening_ratio
def flattening_ratio(self: SymmetricRollPass):
    R_prime = self.flattened_roll_radius
    R       = self.roll.nominal_radius
    h       = self.in_profile.height
    return (R_prime - R) / h
"""

filepath = r"C:\Users\keert\Untitled Folder\pyroll_hitchcock.py"
with open(filepath, "w", encoding="utf-8") as f:
    f.write(plugin_code)

print("Plugin rewritten with UTF-8 encoding ✓")

Plugin rewritten with UTF-8 encoding ✓


In [8]:
# Cell — Import and test our plugin
import importlib.util

spec = importlib.util.spec_from_file_location(
    "pyroll_hitchcock",
    filepath
)
pyroll_hitchcock = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pyroll_hitchcock)

print(f"Plugin loaded ✓  version: {pyroll_hitchcock.VERSION}")
print()

print("New hooks registered:")
print(f"  hitchcock_constant:    {hasattr(pr.RollPass, 'hitchcock_constant')}")
print(f"  flattened_roll_radius: {hasattr(pr.RollPass, 'flattened_roll_radius')}")
print(f"  flattening_ratio:      {hasattr(pr.RollPass, 'flattening_ratio')}")

Plugin loaded ✓  version: 0.1.0

New hooks registered:


NameError: name 'pr' is not defined

In [9]:
# Full Day 6 — all in one cell
import importlib.util
import pyroll.core as pr
import pyroll.wusatowski_spreading
import pyroll.integral_thermal
import numpy as np

# ── Load our plugin ──────────────────────────────────────
filepath = r"C:\Users\keert\Untitled Folder\pyroll_hitchcock.py"
spec = importlib.util.spec_from_file_location("pyroll_hitchcock", filepath)
pyroll_hitchcock = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pyroll_hitchcock)
print(f"Plugin loaded ✓  version: {pyroll_hitchcock.VERSION}")

# ── Verify hooks ─────────────────────────────────────────
print("\nNew hooks registered:")
print(f"  hitchcock_constant:    {hasattr(pr.RollPass, 'hitchcock_constant')}")
print(f"  flattened_roll_radius: {hasattr(pr.RollPass, 'flattened_roll_radius')}")
print(f"  flattening_ratio:      {hasattr(pr.RollPass, 'flattening_ratio')}")

# ── Profile and sequence ─────────────────────────────────
in_profile = pr.Profile.round(
    radius=30e-3,
    temperature=1200 + 273.15,
    strain=0,
    material=["steel", "C15"],
    flow_stress=100e6,
    density=7.5e3,
    specific_heat_capacity=690,
    thermal_conductivity=23,
    environment_temperature=20 + 273.15,
    surface_heat_transfer_coefficient=100,
)

sequence = pr.PassSequence([
    pr.RollPass(
        label="R1 - Oval",
        roll=pr.Roll(
            groove=pr.CircularOvalGroove(r1=2e-3, r2=50e-3, depth=10e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=8e-3,
    ),
    pr.Transport(label="R1->R2", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R2 - Round",
        roll=pr.Roll(
            groove=pr.FalseRoundGroove(r1=2e-3, r2=25e-3, depth=12e-3, flank_angle=30),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=12e-3,
    ),
    pr.Transport(label="R2->R3", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R3 - Oval",
        roll=pr.Roll(
            groove=pr.CircularOvalGroove(r1=2e-3, r2=30e-3, depth=6e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=10e-3,
    ),
    pr.Transport(label="R3->R4", duration=2,
                 environment_temperature=20+273.15,
                 surface_heat_transfer_coefficient=10),
    pr.RollPass(
        label="R4 - Round",
        roll=pr.Roll(
            groove=pr.RoundGroove(r1=1e-3, r2=35e-3, depth=4e-3),
            nominal_radius=155e-3,
            rotational_frequency=1,
            temperature=20 + 273.15,
        ),
        gap=10e-3,
    ),
])

sequence.solve(in_profile)
print("\nSequence solved ✓")

# ── Test our plugin hooks ────────────────────────────────
print("\nPlugin Hook Results:")
print("="*70)
print(f"{'Pass':<15} {'C_H':>12} {'R nom(mm)':>12} {'R flat(mm)':>12} {'Δ/h (-)':>10}")
print("="*70)

for unit in sequence:
    if isinstance(unit, pr.RollPass):
        C_H    = unit.hitchcock_constant
        R_nom  = unit.roll.nominal_radius * 1000
        R_flat = unit.flattened_roll_radius * 1000
        ratio  = unit.flattening_ratio
        print(f"{unit.label:<15} {C_H:>12.2e} {R_nom:>12.1f} {R_flat:>12.1f} {ratio:>10.4f}")

print("="*70)

Plugin loaded ✓  version: 0.1.0

New hooks registered:
  hitchcock_constant:    True
  flattened_roll_radius: True
  flattening_ratio:      True

Sequence solved ✓

Plugin Hook Results:
Pass                     C_H    R nom(mm)   R flat(mm)    Δ/h (-)
R1 - Oval           2.21e-11        155.0        155.5     0.0081
R2 - Round          2.21e-11        155.0        155.5     0.0066
R3 - Oval           2.21e-11        155.0        155.7     0.0160
R4 - Round          2.21e-11        155.0        155.8     0.0168
